<a href="https://colab.research.google.com/github/theaok/hfgi/blob/main/hfgi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#---------------------------SETUP----------------------------------
#get useful libraries
import io, time, os, sys, re #basics
import zipfile, json, datetime, string   #string for annotating points in scatter
import urllib.request
import numpy as np #basic math
from statistics import * #stats

import matplotlib.pyplot as plt #import pylab as plt #apparently discouraged now:
 #https://stackoverflow.com/questions/11469336/what-is-the-difference-between-pylab-and-pyplot
 #https://www.tutorialspoint.com/matplotlib/matplotlib_pylab_module.htm

import pandas as pd
import pandas_datareader as pdr
from pandas_datareader import wb
from pandas.io.formats.style import Styler
#s4 = Styler(df4, uuid_len=0, cell_ids=False)

import urllib  #weird, guess need to have os and pandas imported for this to work  %TODO/LATER ditch it, its weird anyway, just use wget/curl

from google.colab import files

#import webbrowser

import seaborn as sns

from google.colab import data_table
data_table.enable_dataframe_formatter() #this enables spreadsheet view upon calling dataframe (without() )

#many tricks how to extend notebook functionality
#https://coderzcolumn.com/tutorials/python/list-of-useful-magic-commands-in-jupyter-notebook-lab
#will display all output not just last command
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

#MAGICS and THEMES/STYLES: important! does affect not just shading/colors, but also fonts, spacing, etc
#(even if you only select default (v not selecting anything) [but does seem to work better if you do make explicit sleections])

###magics: https://ipython.readthedocs.io/en/stable/interactive/magics.html
#most essential setup for vis: it does affect vis! careful!! stick with inline, maybe notebook; others mostly for non-notebook, eg spyder environ
#https://jakevdp.github.io/PythonDataScienceHandbook/04.00-introduction-to-matplotlib.html recomends *inline*!
#show current one:
#%matplotlib
#%matplotlib --list
#interactive plots:
#%matplotlib notebook
#static images of your plot:
%matplotlib inline
#may play with this one and other magics (btw default is probably agg)
#%matplotlib nbagg
##https://www.marktechpost.com/2023/10/20/6-magic-commands-for-jupyter-notebooks-in-python-data-science/
#%%latex
#%ai
#%run
#%writefile
#%history -n

###themes/styles: https://matplotlib.org/stable/gallery/style_sheets/style_sheets_reference.html
#https://jakevdp.github.io/PythonDataScienceHandbook/04.11-settings-and-stylesheets.html
#https://matplotlib.org/stable/tutorials/introductory/customizing.html
#here more about art and style than under the hood functionality as with magics, explore and experiment
#many may find 'default' or seaborn ones more pleasing; my fav 'classic' is back from 90s ;)
#plt.style.available #list available styles :) may install more
#plt.style.use('default') # more delicate subtle than classic
plt.style.use('classic')  #  'seaborn-whitegrid' 'seaborn-white' 'seaborn-poster'
# btw: magics v theme/style sequence matters, eg if i specify classic style before inline magic, i wouldnt get grey bounding box im getting

#sometimes have to install library which you get from https://pypi.org/
#!pip install geopandas

import statsmodels.formula.api as sm

https://dataverse.harvard.edu/file.xhtml?fileId=13094135&version=3.1

using
flourishingCountyYear.csv

TODO invite some of the authors of the data start with
Iacus, Stefano Maria and if not ask to rec co-author

>>>see how many obs per county min if few collapse over yrs!
then just descriptive stats and map and pop den like in brian everett paper

In [3]:
df=pd.read_csv('https://rutgers.box.com/shared/static/l7f4uiev8ju8b6xho2w0tbeplqdkxfle.zip')

/tmp/ipykernel_2633/2070100474.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('https://rutgers.box.com/shared/static/l7f4uiev8ju8b6xho2w0tbeplqdkxfle.zip')


In [4]:
df.shape

(2340928, 11)

In [5]:
df.columns

Index(['variable', 'stat', 'stat_se', 'salience', 'ntweets', 'validtweets',
       'natweets', 'FIPS', 'county', 'StateCounty', 'year'],
      dtype='object')

In [11]:
df.head(15)

,variable,stat,stat_se,salience,ntweets,validtweets,natweets,FIPS,county,StateCounty,year
0,loneliness,NaN,NaN,0.00,1,0,1,36.00,119.00,36119,2010
1,charity,NaN,NaN,0.00,1,0,1,6.00,83.00,06083,2010
2,jobsat,-1.00,NaN,1.00,1,1,0,31.00,55.00,31055,2010
3,purpose,-1.00,NaN,1.00,1,1,0,31.00,55.00,31055,2010
4,optimism,1.00,NaN,1.00,1,1,0,55.00,79.00,55079,2010
5,drinkhealth,NaN,NaN,0.00,1,0,1,48.00,451.00,48451,2011
6,healthlim,1.00,NaN,0.50,2,1,1,34.00,17.00,34017,2011
7,volunteer,NaN,NaN,0.00,1,0,1,18.00,57.00,18057,2011
8,optimism,NaN,NaN,0.00,1,0,1,31.00,119.00,31119,2011
9,goodpromo,NaN,NaN,0.00,1,0,1,37.00,81.00,37081,2011


In [13]:
pd.set_option('display.max_rows', None)
df.variable.value_counts() #lifesat for now

,count
variable,
loneliness,37873
charity,37873
jobsat,37873
purpose,37873
optimism,37873
drinkhealth,37873
healthlim,37873
volunteer,37873
goodpromo,37873


In [10]:
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
df.ntweets.describe()

,ntweets
count,2340928.00
mean,61449.18
std,386376.78
min,1.00
25%,208.00
50%,2129.00
75%,15517.25
max,32699771.00


In [14]:
df = df[df['variable'] == 'lifesat']

In [17]:
pd.set_option('display.max_rows', None)
df.ntweets.describe()

,ntweets
count,37873.00
mean,69057.64
std,409042.14
min,1.00
25%,485.00
50%,3203.00
75%,19822.00
max,32699771.00


In [23]:
df = df[df['ntweets'] > 500] #do note validtweets is as low as >65 and <100 for like 20obs
df.shape
df.year.value_counts()

(28265, 11)

,count
year,
2014,3101
2013,3079
2015,2945
2018,2595
2019,2551
2020,2468
2017,2465
2021,2370
2022,2257


In [21]:
df.head(1000)

,variable,stat,stat_se,salience,ntweets,validtweets,natweets,FIPS,county,StateCounty,year
84,lifesat,-0.20,0.69,0.35,1101,367,734,28.00,67.00,28067,2012
779,lifesat,-0.20,0.09,0.37,82345,30352,51993,26.00,149.00,26149,2013
780,lifesat,-0.19,0.14,0.38,31628,11708,19920,39.00,135.00,39135,2013
781,lifesat,-0.19,0.11,0.37,68801,24905,43896,21.00,179.00,21179,2013
782,lifesat,-0.20,0.14,0.36,38191,13621,24570,22.00,75.00,22075,2013
783,lifesat,-0.18,0.14,0.36,37392,13199,24193,1.00,31.00,01031,2013
1111,lifesat,-0.20,0.35,0.37,5517,2005,3512,20.00,205.00,20205,2013
1112,lifesat,-0.18,0.25,0.36,10998,3992,7006,27.00,149.00,27149,2013
1113,lifesat,-0.20,0.13,0.35,45787,15985,29802,31.00,119.00,31119,2013
1114,lifesat,-0.13,0.29,0.38,7148,2579,4569,53.00,69.00,53069,2013


In [24]:
df = df[df['year'] <= 2019] #drop covid years

so have 12-19 just 8yrs reasonable to take avg and then get pop siz and density for mid yr ie 2016 (closer to 19 fine bc 12 had fewer obs)